# `02_r5_descriptive.ipynb` — R5 PRIMARY: Variance decomposition of $|\Delta\ln \text{NotionalCost}^{COP}|$ via stationary bootstrap

**Spec citations:** `docs/specs/2026-05-16-ai-cost-factor-model-design.md` v0.2.5 — §2.1 (R5 PRIMARY outputs 1–5), §0.1 CORRECTIONS-P (stationary bootstrap Politis-Romano with expected block length $\lceil T^{1/3}\rceil$, $B=10{,}000$), CORRECTIONS-Q (conservative covariance attribution — cov reported separately, NOT in FX-share denominator), CORRECTIONS-R (R5 Role A — INDIVIDUAL-sizing only), CORRECTIONS-T (CI half-width threshold $\leq 0.15$ on FX share; M-design usability gate, NOT verdict-bearing), §0.3 CORRECTIONS-Y-6 ($\hat\pi$ ephemeral-cost disclosure).

## Headline disclaimer (CORRECTIONS-R Role A — pinned)

> This notebook reports the FX channel's contribution to the variance of COP-denominated notional cost-burden risk for the proxy usage pattern. **It does NOT claim a causal channel.** The output supports M-design sizing for **INDIVIDUAL** hedge instruments — i.e., "for this notional USD burn at this TRM exposure, the hedge position must absorb $Y$ COP of monthly cost-burden variance." It does NOT establish population-scale FX-vol-on-cost transmission. Channel-existence claims require the R4-S3-USD behavioral arm (Task 14/15), which is independent of R5.

## $\hat\pi$ disclosure (CORRECTIONS-Y-6 — pinned)

> Cost magnitudes throughout the panel apply the 5m-tier rate to total cache-creation tokens. True economic cost is up to $1 + \hat\pi\cdot(\text{rate}_{1h}/\text{rate}_{5m} - 1) \approx 1.40\times$ higher ($\hat\pi \approx 0.399$ from the panel; $\text{rate}_{1h} \approx 2\cdot\text{rate}_{5m}$). **The variance-based FX-share output is unaffected by this multiplicative bias** — the bias cancels in the ratio $\text{Var}(\Delta\ln \text{USDCOP})/\text{Var}(\Delta\ln \text{NotionalCost}^{COP})$. Absolute σ̂ and VaR magnitudes ARE biased downward by up to ~40% in level terms.

## Anti-fishing pins (set PRE-DATA — DO NOT TUNE)

| Pin | Value | Source |
|---|---|---|
| Bootstrap reps $B$ | `10_000` | CORRECTIONS-P |
| Expected block length | $\lceil T^{1/3}\rceil$, where $T = 28$ (after first-diff of $N=29$) → block_len = `4` | CORRECTIONS-P |
| Confidence level | `0.90` (two-sided) | §2.1 outputs |
| Bootstrap method | `"basic"` | `arch.bootstrap` default |
| Seed | `20260516` | pre-pinned |
| CI half-width threshold on FX share | $\leq 0.15$ — **M-design usability only, NOT verdict-bearing** | CORRECTIONS-T |

## Inputs

- Production panel `data/panels/notional_cost_panel.parquet` (29 weekday-UTC rows, 2026-01-06 → 2026-05-14, ccusage-parity 0.9994×, sha256 `c002df3b…`).
- Task 11 power-HALT verdict: **DID NOT FIRE** (power = 0.7115 ≥ 0.50 at MDES = 0.40 residual-SD). R5 is descriptive and NOT power-gated; this notebook proceeds independently.


## Trio 1 — Load panel + compute $\Delta\ln$ series

### Decision-citation block (4-part)

- **Reference:** spec §3.5 `DailyNotionalPanel` contract + `DATA_PROVENANCE.md`.
- **Why:** the parquet is the canonical Tier-3 artifact at sha256 `c002df3b…`; all subsequent $\Delta\ln$ series are deterministic derivatives of its `notional_cost_cop`, `notional_cost_usd`, and `trm_cop_per_usd` columns. The first-difference of the natural log is the canonical pre-transform for variance decomposition under the multiplicative identity $\text{NotionalCost}^{COP}_t = \text{NotionalCost}^{USD}_t \cdot \text{USDCOP}_t$ (which gives $\Delta\ln \text{Cost}^{COP} = \Delta\ln \text{Cost}^{USD} + \Delta\ln \text{USDCOP}$ exactly per row).
- **Relevance:** Politis & Romano stationary bootstrap operates on a stationary 1-D series; first-differencing the log enforces approximate stationarity on the multiplicatively-growing levels.
- **Connection:** feeds Trios 2–6 (every downstream test consumes `dln_cop`, `dln_usd`, `dln_trm`).

### Why

Load the production panel, compute three $\Delta\ln$ series, and verify the additive identity $\Delta\ln \text{Cost}^{COP} \approx \Delta\ln \text{Cost}^{USD} + \Delta\ln \text{USDCOP}$ (algebraic equality up to floating-point) before any bootstrap work.


In [1]:
from __future__ import annotations

import math
from pathlib import Path

import numpy as np
import polars as pl
from arch.bootstrap import StationaryBootstrap

PANEL_PATH = Path('../../data/panels/notional_cost_panel.parquet')

df = pl.read_parquet(PANEL_PATH)
print('shape:', df.shape)
print('date window:', df['date_utc'].min(), '->', df['date_utc'].max())

# Derive Δln series (natural log, first difference)
dln_cop = np.diff(np.log(df['notional_cost_cop'].to_numpy()))
dln_usd = np.diff(np.log(df['notional_cost_usd'].to_numpy()))
dln_trm = np.diff(np.log(df['trm_cop_per_usd'].to_numpy()))

T = len(dln_cop)
print(f'\nT (length of Δln series, after first-diff) = {T}')
print(f'NaN check — dln_cop: {int(np.isnan(dln_cop).sum())}, dln_usd: {int(np.isnan(dln_usd).sum())}, dln_trm: {int(np.isnan(dln_trm).sum())}')

print('\nFirst 5 values of each Δln series:')
print(f'  dln_cop: {np.array2string(dln_cop[:5], precision=6)}')
print(f'  dln_usd: {np.array2string(dln_usd[:5], precision=6)}')
print(f'  dln_trm: {np.array2string(dln_trm[:5], precision=6)}')

print('\nSummary stats:')
for name, s in [('dln_cop', dln_cop), ('dln_usd', dln_usd), ('dln_trm', dln_trm)]:
    print(f'  {name}: mean={s.mean():+.5f}, sd={s.std(ddof=1):.5f}, min={s.min():+.5f}, max={s.max():+.5f}')

# Additive identity check (should hold up to FP error)
identity_err = np.max(np.abs(dln_cop - (dln_usd + dln_trm)))
print(f'\nAdditive-identity max abs error  max|dln_cop − (dln_usd + dln_trm)| = {identity_err:.2e}')
assert identity_err < 1e-10, 'Δln identity broken — check panel construction'
print('Identity holds (algebraic).')


shape: (29, 11)
date window: 2026-01-06 -> 2026-05-14

T (length of Δln series, after first-diff) = 28
NaN check — dln_cop: 0, dln_usd: 0, dln_trm: 0

First 5 values of each Δln series:
  dln_cop: [0.121438 2.397409 0.925088 1.060795 1.168635]
  dln_usd: [0.132043 2.387149 0.917446 1.073043 1.164238]
  dln_trm: [-0.010605  0.01026   0.007642 -0.012248  0.004397]

Summary stats:
  dln_cop: mean=+0.02038, sd=1.65553, min=-5.47857, max=+2.86498
  dln_usd: mean=+0.02014, sd=1.65642, min=-5.48382, max=+2.87524
  dln_trm: mean=+0.00023, sd=0.00881, min=-0.01287, max=+0.02339

Additive-identity max abs error  max|dln_cop − (dln_usd + dln_trm)| = 1.78e-15
Identity holds (algebraic).


### Interpretation

$T = 28$ after first-difference (panel $N=29$ → 28 differences). All three series are NaN-free. The additive identity $\Delta\ln \text{Cost}^{COP} = \Delta\ln \text{Cost}^{USD} + \Delta\ln \text{USDCOP}$ holds to machine precision, confirming the panel's multiplicative construction is internally consistent. The TRM series has small magnitude (low daily FX vol over this window), while `dln_usd` and `dln_cop` are large-magnitude due to bursty single-developer usage. All values are finite and well-formed.


## Trio 2 — Realized vol of $\Delta\ln \text{NotionalCost}^{COP}$ with 90% stationary-bootstrap CI

### Decision-citation block (4-part)

- **Reference:** spec §2.1 output 1 + §0.1 CORRECTIONS-P.
- **Why:** stationary bootstrap (Politis & Romano 1994) is the canonical serial-correlation-robust resampler for short-T variance/quantile statistics. IID bootstrap is explicitly rejected by CORRECTIONS-P because it destroys the within-block dependence that drives the variance of variance. `arch.bootstrap.StationaryBootstrap` implements the geometric block-length scheme with expected block length passed as the first argument.
- **Relevance:** the headline σ̂ feeds the M-design hedge sizing under Role A (CORRECTIONS-R) and the FX-share denominator in Trio 4.
- **Connection:** block length $\lceil T^{1/3}\rceil = \lceil 28^{1/3}\rceil = \lceil 3.04\rceil = 4$ derived from $T$ in Trio 1; $B = 10{,}000$ and seed = 20260516 are pre-data ANTI-FISHING PINS.

### Why

Compute $\hat\sigma = \sqrt{\widehat{\text{Var}}(\Delta\ln \text{NotionalCost}^{COP})}$ from the full sample and obtain its 90% stationary-bootstrap CI.


In [2]:
# Pinned constants — DO NOT TUNE
B = 10_000
SEED = 20260516
ALPHA_SIZE = 0.90  # 90% CI
METHOD = 'basic'
BLOCK_LEN = math.ceil(T ** (1.0 / 3.0))
print(f'PINS: B={B}, seed={SEED}, size={ALPHA_SIZE}, method={METHOD!r}, block_len=ceil(T^(1/3))=ceil({T**(1/3):.3f})={BLOCK_LEN}')


def realized_vol(x: np.ndarray) -> float:
    """Population (ddof=0) standard deviation — spec-consistent with variance decomposition."""
    return float(np.sqrt(np.var(x)))


sigma_hat = realized_vol(dln_cop)
sb_sigma = StationaryBootstrap(BLOCK_LEN, dln_cop, seed=SEED)
ci_sigma = sb_sigma.conf_int(realized_vol, reps=B, size=ALPHA_SIZE, method=METHOD)

ci_lo, ci_hi = float(ci_sigma[0, 0]), float(ci_sigma[1, 0])
ci_half = (ci_hi - ci_lo) / 2.0
print(f'\nσ̂ realized = {sigma_hat:.6f}')
print(f'90% bootstrap CI = [{ci_lo:.6f}, {ci_hi:.6f}]')
print(f'CI half-width = {ci_half:.6f}; ratio half/σ̂ = {ci_half/sigma_hat:.4f}')


PINS: B=10000, seed=20260516, size=0.9, method='basic', block_len=ceil(T^(1/3))=ceil(3.037)=4

σ̂ realized = 1.625700
90% bootstrap CI = [1.216429, 2.124685]
CI half-width = 0.454128; ratio half/σ̂ = 0.2793


### Interpretation

$\hat\sigma$ is the realized population SD of the daily log-return of COP-denominated notional cost burden. The 90% stationary-bootstrap CI accounts for short-range serial dependence within blocks of expected length 4. The ratio (CI half-width)/σ̂ is the relative precision of the σ̂ estimate. **Note (CORRECTIONS-Y-6):** the absolute magnitude is biased downward by up to ~40% due to the 5m-rate-on-total-cache-creation construction; the *ratio*-based FX share in Trio 4 is unaffected.


## Trio 3 — Empirical 5% (daily) VaR of $\Delta\ln \text{NotionalCost}^{COP}$ with 90% bootstrap CI

### Decision-citation block (4-part)

- **Reference:** spec §2.1 output 2.
- **Why:** 5% VaR maps directly to M-design tail-risk sizing per Role A — the hedge must cover one-day adverse moves at the 5th-percentile threshold. Empirical quantile (rather than parametric Normal/Student-t) avoids tail-shape assumptions on a small sample.
- **Relevance:** spec output 2 specifies the 5% VaR; we report the **daily** empirical quantile (raw $\Delta\ln \text{Cost}^{COP}$ at $p=0.05$). The spec's "monthly" framing is conceptual — over the 4.5-month panel, the 5th percentile of the daily distribution is the apples-to-apples sample-quantile statistic given $T=28$. A separate monthly aggregation would require T~22 non-overlapping monthly buckets, which the panel does not support.
- **Connection:** same stationary-bootstrap machinery (seed, block_len, B, method) as Trio 2 — consistency under the ANTI-FISHING PINS.

### Why

Compute the empirical 5th percentile of daily $\Delta\ln \text{NotionalCost}^{COP}$ and its 90% stationary-bootstrap CI.


In [3]:
def var_5pct(x: np.ndarray) -> float:
    """Empirical 5% lower quantile (left tail) — adverse move on cost burden."""
    return float(np.quantile(x, 0.05))


var5_hat = var_5pct(dln_cop)
sb_var = StationaryBootstrap(BLOCK_LEN, dln_cop, seed=SEED)
ci_var = sb_var.conf_int(var_5pct, reps=B, size=ALPHA_SIZE, method=METHOD)

var_lo, var_hi = float(ci_var[0, 0]), float(ci_var[1, 0])
var_half = (var_hi - var_lo) / 2.0
print(f'VaR_5 (daily, empirical) = {var5_hat:.6f}')
print(f'90% bootstrap CI         = [{var_lo:.6f}, {var_hi:.6f}]')
print(f'CI half-width            = {var_half:.6f}')


VaR_5 (daily, empirical) = -2.556998
90% bootstrap CI         = [-3.742679, 0.364579]
CI half-width            = 2.053629


### Interpretation

The 5% daily VaR is the empirical left-tail quantile of $\Delta\ln \text{Cost}^{COP}$: on the worst 5% of days, COP cost-burden change is more negative (i.e., more adverse) than this value. The 90% bootstrap CI is wide relative to the point estimate (typical for quantile estimators at $T=28$). **The hedge sizing under Role A uses σ̂ (Trio 2) as the primary input; VaR is a secondary tail-risk diagnostic.**


## Trio 4 — Variance decomposition (CORRECTIONS-Q conservative covariance attribution)

### Decision-citation block (4-part)

- **Reference:** spec §2.1 output 3 + §0.1 CORRECTIONS-Q.
- **Why:** the additive identity $\text{Var}(\Delta\ln \text{Cost}^{COP}) = \text{Var}(\Delta\ln \text{Cost}^{USD}) + \text{Var}(\Delta\ln \text{USDCOP}) + 2\,\text{Cov}(\Delta\ln \text{Cost}^{USD}, \Delta\ln \text{USDCOP})$ contains an ambiguous covariance term. CORRECTIONS-Q pins the **conservative** rule: report covariance **separately** and EXCLUDE it from both shares. Headline FX share = $\text{Var}(\Delta\ln \text{USDCOP}) / \text{Var}(\Delta\ln \text{Cost}^{COP})$. This prevents over-attribution to the FX channel under positive covariance (which would inflate the apparent FX share if rolled in).
- **Relevance:** the FX share is the headline output that the M-design step consumes for hedge sizing. The conservative rule guarantees the reported FX share is a lower bound on the FX channel's variance contribution under any reasonable attribution scheme.
- **Connection:** stationary bootstrap with **three input series** (`dln_cop`, `dln_usd`, `dln_trm`) — `arch.bootstrap.StationaryBootstrap` resamples them jointly preserving cross-series alignment per row; block_len, B, seed, size, method all carry forward from Trios 2–3.

### Why

Compute three numbers + their 90% stationary-bootstrap CIs:
1. FX share (excluding cov) = $\text{Var}(\Delta\ln \text{USDCOP}) / \text{Var}(\Delta\ln \text{Cost}^{COP})$
2. Usage share (excluding cov) = $\text{Var}(\Delta\ln \text{Cost}^{USD}) / \text{Var}(\Delta\ln \text{Cost}^{COP})$
3. $2\cdot\text{Cov}(\Delta\ln \text{Cost}^{USD}, \Delta\ln \text{USDCOP})$ — diagnostic, reported separately


In [4]:
def fx_share_excl_cov(c: np.ndarray, u: np.ndarray, x: np.ndarray) -> float:
    """Var(Δln USDCOP) / Var(Δln Cost^COP) — CORRECTIONS-Q conservative."""
    return float(np.var(x) / np.var(c))


def usage_share_excl_cov(c: np.ndarray, u: np.ndarray, x: np.ndarray) -> float:
    """Var(Δln Cost^USD) / Var(Δln Cost^COP) — CORRECTIONS-Q conservative."""
    return float(np.var(u) / np.var(c))


def two_cov(c: np.ndarray, u: np.ndarray, x: np.ndarray) -> float:
    """2 · Cov(Δln Cost^USD, Δln USDCOP) — diagnostic, reported separately per CORRECTIONS-Q."""
    return float(2.0 * np.cov(u, x, ddof=0)[0, 1])


# Point estimates from full sample
pt_fx = fx_share_excl_cov(dln_cop, dln_usd, dln_trm)
pt_us = usage_share_excl_cov(dln_cop, dln_usd, dln_trm)
pt_cv = two_cov(dln_cop, dln_usd, dln_trm)

# Joint stationary bootstrap on the three aligned series
sb_dec = StationaryBootstrap(BLOCK_LEN, dln_cop, dln_usd, dln_trm, seed=SEED)
ci_fx = sb_dec.conf_int(fx_share_excl_cov, reps=B, size=ALPHA_SIZE, method=METHOD)
ci_us = sb_dec.conf_int(usage_share_excl_cov, reps=B, size=ALPHA_SIZE, method=METHOD)
ci_cv = sb_dec.conf_int(two_cov, reps=B, size=ALPHA_SIZE, method=METHOD)

fx_lo, fx_hi = float(ci_fx[0, 0]), float(ci_fx[1, 0])
us_lo, us_hi = float(ci_us[0, 0]), float(ci_us[1, 0])
cv_lo, cv_hi = float(ci_cv[0, 0]), float(ci_cv[1, 0])

print(f'FX share (excl cov)    = {pt_fx:.6f}  | 90% CI = [{fx_lo:.6f}, {fx_hi:.6f}]')
print(f'Usage share (excl cov) = {pt_us:.6f}  | 90% CI = [{us_lo:.6f}, {us_hi:.6f}]')
print(f'2 · Cov (diagnostic)   = {pt_cv:.6f}  | 90% CI = [{cv_lo:.6f}, {cv_hi:.6f}]')

# Identity check: FX + Usage + 2·Cov/Var(Cost^COP) should ≈ 1
identity_check = pt_fx + pt_us + (pt_cv / float(np.var(dln_cop)))
print(f'\nIdentity check  FX + Usage + 2·Cov/Var(Cost^COP) = {identity_check:.6f} (should ≈ 1.0)')


FX share (excl cov)    = 0.000028  | 90% CI = [-0.000003, 0.000041]
Usage share (excl cov) = 1.001078  | 90% CI = [0.998897, 1.003730]
2 · Cov (diagnostic)   = -0.002924  | 90% CI = [-0.008650, 0.002815]

Identity check  FX + Usage + 2·Cov/Var(Cost^COP) = 1.000000 (should ≈ 1.0)


### Interpretation

The headline FX share excludes the covariance term per CORRECTIONS-Q — this is the **conservative** lower-bound attribution of the FX channel's variance contribution. The covariance is reported separately as a diagnostic: positive ⇒ FX and usage co-move (additional risk; the sum exceeds 1 minus 2·Cov term); negative ⇒ natural hedge effect within the developer's behavior. The identity check confirms the three components reconstruct the total variance ratio to 1.0 (algebraic). The CI on the FX share is the primary precision diagnostic; Trio 7 evaluates it against the CORRECTIONS-T M-design usability threshold.


## Trio 5 — Counterfactual A: FX held constant at sample mean (token-only vol path)

### Decision-citation block (4-part)

- **Reference:** spec §2.1 output 4 + CORRECTIONS-R Role A.
- **Why:** holding TRM at its sample mean isolates the token/model-only vol path — what the COP cost-burden vol would be if FX were frozen. This is the "hedge-NOT-needed" comparator: the residual vol the user faces from token-volume volatility alone.
- **Relevance:** feeds the M-design "no-hedge" scenario comparison. The gap between this counterfactual σ̂ and the realized σ̂ (Trio 2) is the FX channel's contribution to total vol (vs. the variance-share view in Trio 4).
- **Connection:** comparator to Trio 6 (FX-only path). Together they bound the FX channel's vol contribution under two extreme counterfactuals.

### Why

Construct CF cost = `notional_cost_usd × mean(trm)`, compute its $\Delta\ln$ series, and report its population SD.


In [5]:
trm_mean = float(df['trm_cop_per_usd'].mean())
cf_cost_tokonly = df['notional_cost_usd'].to_numpy() * trm_mean
dln_cf_tokonly = np.diff(np.log(cf_cost_tokonly))
sigma_token_only = float(np.sqrt(np.var(dln_cf_tokonly)))

print(f'mean(TRM) over panel = {trm_mean:.4f} COP/USD')
print(f'σ̂ token-only (FX held at mean) = {sigma_token_only:.6f}')
print(f'σ̂ realized (Trio 2)            = {sigma_hat:.6f}')
print(f'ratio σ̂ token-only / σ̂ realized = {sigma_token_only / sigma_hat:.4f}')
print(f'σ̂ realized − σ̂ token-only      = {sigma_hat - sigma_token_only:+.6f}')

# Sanity: dln of (usd × const) should equal dln(usd) up to FP
diff_check = float(np.max(np.abs(dln_cf_tokonly - dln_usd)))
print(f'\nFP check  max|dln_cf_tokonly − dln_usd| = {diff_check:.2e} (should be ~0 by construction)')


mean(TRM) over panel = 3688.6390 COP/USD
σ̂ token-only (FX held at mean) = 1.626576
σ̂ realized (Trio 2)            = 1.625700
ratio σ̂ token-only / σ̂ realized = 1.0005
σ̂ realized − σ̂ token-only      = -0.000876

FP check  max|dln_cf_tokonly − dln_usd| = 1.33e-15 (should be ~0 by construction)


### Interpretation

The token-only counterfactual SD equals $\hat\sigma_{\Delta\ln \text{Cost}^{USD}}$ by construction (multiplying by a constant cancels in the log-difference). The ratio σ̂_token-only / σ̂_realized reveals how much of the realized vol survives when FX is frozen: ratios close to 1 indicate FX adds little vol; ratios far below 1 indicate FX is a meaningful contributor. The difference σ̂_realized − σ̂_token-only is NOT the FX vol contribution (variance isn't linear in SD), but it is a useful sizing reference for the M-design.


## Trio 6 — Counterfactual B: tokens held constant at sample-mean daily usage (FX-only vol path, Role A)

> **CORRECTIONS-R Role A reminder (preamble):** This counterfactual reflects the FX channel's INDIVIDUAL-sizing contribution per Role A. It does NOT establish a transmission channel; the verdict on transmission requires the R4-S3-USD behavioral arm (Task 14), which is an independent inferential test on a different LHS variable. R5 produces a hedge-sizing number; R4-S3-USD produces a channel-existence test. Conflating the two is the v0.1.0 trap that v0.2.5 explicitly avoids.

### Decision-citation block (4-part)

- **Reference:** spec §2.1 output 4 (second bullet) + CORRECTIONS-R Role A.
- **Why:** holding tokens at the sample-mean daily usage isolates the FX-only vol path — informationally equivalent to Banrep's TRM panel scaled by a constant. This is the input to **individual-sizing of the M-design hedge** under Role A.
- **Relevance:** the σ̂ of this counterfactual approximates the FX-vol-induced contribution to cost-burden vol for a "typical-usage" day. Combined with the variance-share number from Trio 4, it gives the M-design step two consistent views of the FX channel's individual-sizing magnitude.
- **Connection:** comparator to Trio 5 (token-only path); both feed the Trio 9 headline.

### Why

Construct CF cost = `mean(notional_cost_usd) × trm_cop_per_usd`, compute its $\Delta\ln$ series, and report its population SD.


In [6]:
usd_mean = float(df['notional_cost_usd'].mean())
cf_cost_fxonly = usd_mean * df['trm_cop_per_usd'].to_numpy()
dln_cf_fxonly = np.diff(np.log(cf_cost_fxonly))
sigma_fx_only = float(np.sqrt(np.var(dln_cf_fxonly)))

print(f'mean(NotionalCost^USD) over panel = {usd_mean:.4f} USD/day')
print(f'σ̂ FX-only (tokens held at mean) = {sigma_fx_only:.6f}')
print(f'σ̂ realized (Trio 2)             = {sigma_hat:.6f}')
print(f'σ̂ token-only (Trio 5)           = {sigma_token_only:.6f}')

# Variance-share cross-check: σ_FX^2 / σ_cop^2 vs. FX share (Trio 4)
var_share_from_cf = sigma_fx_only ** 2 / sigma_hat ** 2
print(f'\nVariance-share cross-check:')
print(f'  (σ̂ FX-only)^2 / (σ̂ realized)^2 = {var_share_from_cf:.6f}')
print(f'  FX share (Trio 4, excl cov)     = {pt_fx:.6f}')
print(f'  (should be equal up to FP — both compute Var(Δln TRM)/Var(Δln Cost^COP))')

# Sanity: dln of (const × trm) should equal dln(trm) up to FP
diff_check_fx = float(np.max(np.abs(dln_cf_fxonly - dln_trm)))
print(f'\nFP check  max|dln_cf_fxonly − dln_trm| = {diff_check_fx:.2e} (should be ~0 by construction)')


mean(NotionalCost^USD) over panel = 96.4320 USD/day
σ̂ FX-only (tokens held at mean) = 0.008653
σ̂ realized (Trio 2)             = 1.625700
σ̂ token-only (Trio 5)           = 1.626576

Variance-share cross-check:
  (σ̂ FX-only)^2 / (σ̂ realized)^2 = 0.000028
  FX share (Trio 4, excl cov)     = 0.000028
  (should be equal up to FP — both compute Var(Δln TRM)/Var(Δln Cost^COP))

FP check  max|dln_cf_fxonly − dln_trm| = 1.78e-15 (should be ~0 by construction)


### Interpretation

The FX-only counterfactual SD equals $\hat\sigma_{\Delta\ln \text{USDCOP}}$ by construction (multiplying by a constant cancels in the log-difference). The variance-share cross-check confirms internal consistency: $(\hat\sigma_{\text{FX-only}})^2 / (\hat\sigma_{\text{realized}})^2$ equals the Trio 4 FX share (excl cov) to floating-point precision — both compute the same ratio by different routes. This counterfactual SD is the **direct M-design input** under Role A: for a typical-usage day, the FX channel alone contributes vol of this magnitude to the COP cost-burden time path.


## Trio 7 — Quality-threshold check (CORRECTIONS-T) — M-design usability gate

### Decision-citation block (4-part)

- **Reference:** spec §2.1 verdict-logic table + §0.1 CORRECTIONS-T.
- **Why:** CORRECTIONS-T pins the CI half-width on the FX share at $\leq 0.15$ (tighter than v0.2.0's $\leq 0.20$) — required for non-coin-flip discrimination at the M-design step. **This is NOT a verdict gate** (R5 has no PASS/FAIL); it is an M-design usability gate: if the CI half-width exceeds 0.15, the FX-share point estimate is too imprecise to be useful for hedge sizing, and the M-design step should down-weight it or wait for more data.
- **Relevance:** anti-fishing — if the threshold is breached, document the breach honestly in the headline; **do NOT enlarge T post-hoc** to chase the threshold (that would be silent fishing on the panel boundary).
- **Connection:** consumes `ci_fx` from Trio 4; emits `meets_threshold` boolean for the Trio 9 headline.

### Why

Compute the CI half-width on the FX share (excl cov) and compare against 0.15.


In [7]:
ci_half_width_fx = (fx_hi - fx_lo) / 2.0
THRESHOLD_T = 0.15
meets_threshold = bool(ci_half_width_fx <= THRESHOLD_T)

print(f'FX-share point estimate = {pt_fx:.6f}')
print(f'90% CI                  = [{fx_lo:.6f}, {fx_hi:.6f}]')
print(f'CI half-width           = {ci_half_width_fx:.6f}')
print(f'CORRECTIONS-T threshold = {THRESHOLD_T:.2f}')
print(f'meets_threshold         = {meets_threshold}')
print()
if meets_threshold:
    print('The FX-share estimate is precise enough for M-design sizing under CORRECTIONS-T.')
else:
    print('The FX-share CI half-width exceeds 0.15.')
    print('Per CORRECTIONS-T this is an M-DESIGN USABILITY signal (NOT a verdict gate):')
    print('  - The point estimate is reported honestly with the wide CI.')
    print('  - M-design should treat the FX share as imprecise and consider waiting for more T.')
    print('  - ANTI-FISHING: T is NOT enlarged post-hoc to chase the threshold;')
    print('    this notebook freezes at T=28 (panel boundary) and documents the breach.')


FX-share point estimate = 0.000028
90% CI                  = [-0.000003, 0.000041]
CI half-width           = 0.000022
CORRECTIONS-T threshold = 0.15
meets_threshold         = True

The FX-share estimate is precise enough for M-design sizing under CORRECTIONS-T.


### Interpretation

The CI half-width relative to the 0.15 threshold informs the M-design step's confidence in the FX-share point estimate. **Anti-fishing reminder:** this is not verdict-bearing — the R5 PRIMARY output is the number-plus-CI itself, not a binary pass/fail. If the threshold is breached, the breach is documented and the operator decides whether to wait for more T (legitimate pivot) before consuming the number for hedge sizing.


## Headline output — R5 PRIMARY (consumed by M-design under Role A)


In [8]:
print('=' * 78)
print('R5 PRIMARY HEADLINE — variance decomposition of |Δln NotionalCost^COP|')
print('Spec v0.2.5 §2.1 outputs 1–5  |  Stationary bootstrap (Politis-Romano)')
print(f'Pins: B={B}, block_len={BLOCK_LEN}, seed={SEED}, size={ALPHA_SIZE}, method={METHOD!r}, T={T}')
print('=' * 78)

print('\n[1] Realized vol of Δln NotionalCost^COP')
print(f'    σ̂ = {sigma_hat:.6f}')
print(f'    90% CI = [{ci_lo:.6f}, {ci_hi:.6f}]  (half-width {(ci_hi-ci_lo)/2:.6f})')

print('\n[2] 5% daily VaR of Δln NotionalCost^COP')
print(f'    VaR_5 = {var5_hat:.6f}')
print(f'    90% CI = [{var_lo:.6f}, {var_hi:.6f}]  (half-width {(var_hi-var_lo)/2:.6f})')

print('\n[3] Variance decomposition (CORRECTIONS-Q conservative — cov SEPARATE)')
print(f'    FX share (excl cov)    = {pt_fx:.6f}  | 90% CI = [{fx_lo:.6f}, {fx_hi:.6f}]')
print(f'    Usage share (excl cov) = {pt_us:.6f}  | 90% CI = [{us_lo:.6f}, {us_hi:.6f}]')
print(f'    2 · Cov (diagnostic)   = {pt_cv:.6f}  | 90% CI = [{cv_lo:.6f}, {cv_hi:.6f}]')

print('\n[4] Counterfactual scenarios (Role A — INDIVIDUAL-sizing)')
print(f'    σ̂ token-only (FX held at mean TRM={trm_mean:.2f}) = {sigma_token_only:.6f}')
print(f'    σ̂ FX-only  (tokens held at mean USD={usd_mean:.2f}) = {sigma_fx_only:.6f}')
print(f'    σ̂ realized                                          = {sigma_hat:.6f}')

print('\n[5] M-design usability gate (CORRECTIONS-T)')
print(f'    CI half-width on FX share = {ci_half_width_fx:.6f}  vs. threshold 0.15')
print(f'    meets_threshold = {meets_threshold}  (NOT verdict-bearing; M-design sizing informational)')

print('\n' + '-' * 78)
print('DISCLAIMERS (carried into M-design)')
print('-' * 78)
print('CORRECTIONS-R Role A: R5 quantifies the FX channel\'s contribution to')
print('  INDIVIDUAL-hedge sizing for this proxy usage pattern. It does NOT')
print('  establish a population-scale transmission channel. Channel existence')
print('  requires R4-S3-USD behavioral evidence (Tasks 14/15), which is an')
print('  independent inferential test on a different LHS.')
print()
print('CORRECTIONS-Y-6 π̂ disclaimer: Cost magnitudes apply the 5m-rate to total')
print('  cache-creation tokens; true economic cost is up to ~1.40× higher')
print('  (π̂ ≈ 0.399). The variance-based FX-share IS NOT affected by this bias')
print('  (cancels in the ratio). The absolute σ̂ and VaR magnitudes ARE biased')
print('  downward in level terms.')
print()
print('Anti-fishing: B, seed, block_len, size, method all pinned PRE-DATA per')
print('  CORRECTIONS-P. NO post-hoc tuning. CI threshold breach (if any) is')
print('  documented; T is NOT enlarged to chase the threshold.')
print('=' * 78)


R5 PRIMARY HEADLINE — variance decomposition of |Δln NotionalCost^COP|
Spec v0.2.5 §2.1 outputs 1–5  |  Stationary bootstrap (Politis-Romano)
Pins: B=10000, block_len=4, seed=20260516, size=0.9, method='basic', T=28

[1] Realized vol of Δln NotionalCost^COP
    σ̂ = 1.625700
    90% CI = [1.216429, 2.124685]  (half-width 0.454128)

[2] 5% daily VaR of Δln NotionalCost^COP
    VaR_5 = -2.556998
    90% CI = [-3.742679, 0.364579]  (half-width 2.053629)

[3] Variance decomposition (CORRECTIONS-Q conservative — cov SEPARATE)
    FX share (excl cov)    = 0.000028  | 90% CI = [-0.000003, 0.000041]
    Usage share (excl cov) = 1.001078  | 90% CI = [0.998897, 1.003730]
    2 · Cov (diagnostic)   = -0.002924  | 90% CI = [-0.008650, 0.002815]

[4] Counterfactual scenarios (Role A — INDIVIDUAL-sizing)
    σ̂ token-only (FX held at mean TRM=3688.64) = 1.626576
    σ̂ FX-only  (tokens held at mean USD=96.43) = 0.008653
    σ̂ realized                                          = 1.625700

[5] M-desig

### Final disclaimers (carried into M-design)

> **CORRECTIONS-R Role A (pinned, repeat):** The headline FX share and counterfactual SDs above quantify the FX channel's contribution to **INDIVIDUAL-hedge sizing** for the proxy usage pattern. R5 does NOT establish a population-scale transmission channel. Channel existence requires the R4-S3-USD behavioral arm (Tasks 14/15) — an independent inferential test on $|\Delta\ln \text{NotionalCost}^{USD}_t|$ as LHS with FX-vol regressors. The two outputs answer different questions and are not interchangeable.

> **CORRECTIONS-Y-6 $\hat\pi$ disclaimer (pinned, repeat):** The cost panel applies the 5m-tier rate to total cache-creation tokens; true economic cost is up to $1 + \hat\pi(r_{1h}/r_{5m}-1) \approx 1.40\times$ higher with $\hat\pi \approx 0.399$. **The variance-based FX-share output is unaffected** (multiplicative bias cancels in ratios). Absolute σ̂ and VaR magnitudes are biased downward in level terms; M-design sizing should apply a conservative inflation factor of up to 1.40× to absolute COP-vol numbers consumed from this notebook.

> **Anti-fishing summary:** $B = 10{,}000$, seed = `20260516`, block_len = $\lceil T^{1/3}\rceil = 4$, size = 0.90, method = `"basic"`, conservative covariance attribution (CORRECTIONS-Q), CI threshold = 0.15 (CORRECTIONS-T, NOT verdict-bearing) — all pinned PRE-DATA. No post-hoc tuning was performed. The CI threshold check is informational only; $T$ was not enlarged to chase the threshold.
